<a href="https://colab.research.google.com/github/CodeHunterOfficial/ABCD_ASPNETCORE/blob/main/NLP-2026/Lecture_2/Doc2Vec_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Doc2Vec: эмбеддинги документов

## Введение: почему Doc2Vec заслуживает отдельного разбора

Мы прошли долгий путь от one-hot encoding до FastText. Все эти методы решают одну задачу: представить **слово** в виде плотного вектора. Но в реальных NLP-задачах — классификация текстов, кластеризация, информационный поиск, рекомендательные системы — нам часто нужно представить **целый документ**, а не отдельное слово. И тут возникает фундаментальный вопрос: как из векторов слов получить вектор документа?

Самый простой подход — **усреднить векторы слов документа**:

$$
u_d = \frac{1}{|d|} \sum_{w \in d} u_w.
$$

Этот подход работает на удивление хорошо и до сих пор используется как baseline. Но у него есть принципиальные ограничения:

1. **Потеря порядка слов.** Усреднение игнорирует порядок слов, как BoW. Фразы «кошка сидит на собаке» и «собака сидит на кошке» дадут одинаковый вектор.

2. **Потеря информации о редких словах.** Если документ содержит редкое, но важное слово, его вклад в среднее может быть потерян среди частых слов.

3. **Нет обучения на уровне документа.** Вектор документа — это просто производная от векторов слов. Модель не учится различать документы напрямую.

4. **Нет учёта глобального контекста документа.** Усреднение не различает документы, которые используют одни и те же слова в разных смыслах.

Doc2Vec, предложенный Томашем Миколовым и Квоком Ле в 2014 году (те же авторы, что и Word2Vec), решает эти проблемы элегантным способом: **мы добавляем вектор документа в модель Word2Vec и обучаем его так же, как обучаем векторы слов**.

Идея проста: в модели появляется новый параметр — вектор документа $d$. Он участвует в предсказании контекста наравне со словами. В процессе обучения модель вынуждена настраивать вектор документа так, чтобы он помогал предсказывать слова этого документа. В результате вектор документа кодирует **тематику** и **содержание** документа.

Ключевое преимущество Doc2Vec перед усреднением: вектор документа **обучается** на задаче предсказания, а не просто вычисляется. Это позволяет ему улавливать более тонкие семантические связи.

В этой лекции мы подробно разберём:

- что такое Doc2Vec и как он связан с Word2Vec;
- две архитектуры: PV-DM и PV-DBOW;
- полный вывод формул для PV-DM;
- полный вывод формул для PV-DBOW;
- градиенты и обновление параметров;
- как обучать Doc2Vec;
- сравнение с усреднением word-векторов;
- свойства и ограничения Doc2Vec;
- практические рекомендации.

---

## 1. От Word2Vec к Doc2Vec

### 1.1 Напоминание: Word2Vec

В Word2Vec у нас есть два набора параметров:

- $U \in \mathbb{R}^{N \times d}$ — входные векторы слов;
- $V \in \mathbb{R}^{N \times d}$ — выходные векторы слов.

Для пары (целевое слово $w_I$, контекстное слово $w_O$) вероятность:

$$
P(w_O \mid w_I) = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})}.
$$

Мы максимизируем правдоподобие по $U$ и $V$.

### 1.2 Идея Doc2Vec

Doc2Vec добавляет **третий набор параметров**: векторы документов $D \in \mathbb{R}^{M \times d}$, где $M$ — число документов, $d$ — размерность. Строка $D_d$ — это вектор документа $d$.

**Ключевая идея:** вектор документа участвует в предсказании слов этого документа наравне со словами. Это означает, что модель должна настроить вектор документа так, чтобы он помогал предсказывать слова, которые в нём встречаются.

**Аналогия:** представьте, что каждый документ — это «тема», а вектор документа — это «описание темы». Модель учится предсказывать слова документа по этому описанию. Если вектор документа хорошо описывает тему, предсказание будет точным.

**Тонкий момент:** в Word2Vec вектор слова $u_w$ — это параметр, который общий для всех вхождений слова в корпус. В Doc2Vec вектор документа $D_d$ — это параметр, который **уникален для каждого документа**. Это означает, что число параметров растёт с числом документов (как в pLSA). Но это не проблема, потому что мы обычно не хотим обобщать на новые документы (для этого есть отдельные методы).

### 1.3 Две архитектуры Doc2Vec

Как и Word2Vec, Doc2Vec имеет две архитектуры:

1. **PV-DM (Paragraph Vector — Distributed Memory):** аналог CBOW. По контексту и вектору документа предсказываем целевое слово.
2. **PV-DBOW (Paragraph Vector — Distributed Bag of Words):** аналог Skip-gram. По вектору документа предсказываем слова документа.

**Тонкий момент:** в оригинальной статье рекомендуется использовать **комбинацию** PV-DM и PV-DBOW: обучать обе модели и конкатенировать их векторы документов. Это даёт лучшее качество, чем каждая модель по отдельности.

Мы разберём обе архитектуры подробно.

---

## 2. PV-DM (Paragraph Vector — Distributed Memory)

### 2.1 Постановка задачи

Пусть дан корпус из $M$ документов. Каждый документ $d$ — это последовательность слов $w_1^{(d)}, w_2^{(d)}, \ldots, w_{T_d}^{(d)}$, где $T_d$ — длина документа.

**Задача PV-DM:** для каждой позиции $t$ в документе $d$ предсказать целевое слово $w_t^{(d)}$ по контексту $C_t^{(d)}$ и вектору документа $D_d$.

**Контекст:** $C_t^{(d)} = \{w_{t-m}^{(d)}, \ldots, w_{t-1}^{(d)}, w_{t+1}^{(d)}, \ldots, w_{t+m}^{(d)}\}$ — слова в окне размера $m$ вокруг целевого слова.

**Ключевое отличие от CBOW:** в CBOW мы предсказываем слово по контексту. В PV-DM мы предсказываем слово по контексту **и** вектору документа. Вектор документа действует как «память» о теме документа, которая помогает предсказывать слова.

### 2.2 Архитектура

PV-DM имеет следующие параметры:

- $U \in \mathbb{R}^{N \times d}$ — входные векторы слов;
- $V \in \mathbb{R}^{N \times d}$ — выходные векторы слов;
- $D \in \mathbb{R}^{M \times d}$ — векторы документов.

**Прямой проход:**

**Шаг 1: получение векторов контекстных слов.**  
Для каждого контекстного слова $c \in C_t^{(d)}$ берём его входной вектор $u_c$.

**Шаг 2: получение вектора документа.**  
Берём вектор документа $D_d$.

**Шаг 3: конкатенация или усреднение.**  
В оригинальной статье используется **конкатенация**:

$$
h = [u_{c_1}; u_{c_2}; \ldots; u_{c_{2m}}; D_d] \in \mathbb{R}^{(2m+1)d}.
$$

Альтернативно можно использовать усреднение:

$$
h = \frac{1}{2m+1} \left( \sum_{c \in C_t^{(d)}} u_c + D_d \right) \in \mathbb{R}^d.
$$

**Тонкий момент:** конкатенация даёт вектор размерности $(2m+1)d$, что увеличивает число параметров и вычислительную сложность. Усреднение даёт вектор размерности $d$, что проще и быстрее. В оригинальной статье используется конкатенация, но на практике часто используют усреднение. Мы будем использовать **усреднение** для простоты, но будем помнить о различии.

**Шаг 4: вычисление оценки для каждого слова.**  
Для каждого слова $w \in V$ вычисляем скалярное произведение:

$$
s_w = v_w^\top h.
$$

**Шаг 5: softmax.**  
Вероятность целевого слова:

$$
P(w_t^{(d)} \mid C_t^{(d)}, D_d) = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

### 2.3 Функция правдоподобия

Для всего корпуса:

$$
\mathcal{L} = \prod_{d=1}^{M} \prod_{t=1}^{T_d} P(w_t^{(d)} \mid C_t^{(d)}, D_d).
$$

Логарифм:

$$
\ell = \sum_{d=1}^{M} \sum_{t=1}^{T_d} \log P(w_t^{(d)} \mid C_t^{(d)}, D_d).
$$

Подставляя выражение для $P$:

$$
\ell = \sum_{d=1}^{M} \sum_{t=1}^{T_d} \left[ v_{w_t}^\top h - \log \sum_{w \in V} \exp(v_w^\top h) \right].
$$

**Цель:** максимизировать $\ell$ по $U, V, D$.

### 2.4 Negative sampling для PV-DM

Как и в Word2Vec, знаменатель softmax требует суммирования по всему словарю. Решение — negative sampling.

Для одной позиции $t$ в документе $d$ с $K$ отрицательными примерами:

$$
\mathcal{L}_{\text{neg}} = \log \sigma(v_{w_t}^\top h) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h).
$$

**Отрицательные примеры** выбираются из шумового распределения $P_n(w) \propto \text{count}(w)^{3/4}$.

### 2.5 Градиенты

Выведем градиенты для одной позиции $t$ в документе $d$ с одним отрицательным примером $w_{\text{neg}}$.

**Обозначения:**

- $h \in \mathbb{R}^d$ — усреднённый вектор контекста и документа;
- $v = v_{w_t}$ — выходной вектор целевого слова;
- $v' = v_{w_{\text{neg}}}$ — выходной вектор отрицательного слова;
- $x = v^\top h$, $x' = v'^\top h$;
- $c_1, \ldots, c_{2m}$ — контекстные слова.

**Функция потерь:**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x').
$$

**Градиент по $v$:**

$$
\frac{\partial \mathcal{L}}{\partial v} = (1 - \sigma(x)) h.
$$

**Градиент по $v'$:**

$$
\frac{\partial \mathcal{L}}{\partial v'} = -\sigma(x') h.
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) v - \sigma(x') v'.
$$

**Градиенты по входным векторам контекстных слов:**

Поскольку $h = \frac{1}{2m+1} \left( \sum_{c \in C_t} u_c + D_d \right)$, градиент по каждому $u_c$:

$$
\frac{\partial \mathcal{L}}{\partial u_c} = \frac{1}{2m+1} \frac{\partial \mathcal{L}}{\partial h}, \quad \forall c \in C_t.
$$

**Градиент по вектору документа $D_d$:**

$$
\frac{\partial \mathcal{L}}{\partial D_d} = \frac{1}{2m+1} \frac{\partial \mathcal{L}}{\partial h}.
$$

**Тонкий момент:** вектор документа обновляется **для каждой позиции** в документе. Это означает, что за одну эпоху вектор документа получает $T_d$ обновлений. Это ключевое отличие от векторов слов, которые получают обновления только когда слово встречается в контексте.

### 2.6 Обновление параметров

Используя градиентный подъём с $\eta$:

$$
v \leftarrow v + \eta (1 - \sigma(x)) h,
$$

$$
v' \leftarrow v' - \eta \sigma(x') h,
$$

$$
u_c \leftarrow u_c + \frac{\eta}{2m+1} \left[ (1 - \sigma(x)) v - \sigma(x') v' \right], \quad \forall c \in C_t,
$$

$$
D_d \leftarrow D_d + \frac{\eta}{2m+1} \left[ (1 - \sigma(x)) v - \sigma(x') v' \right].
$$

**Интерпретация:**

- Вектор документа обновляется в направлении, которое увеличивает вероятность правильного целевого слова. Это означает, что вектор документа «запоминает» информацию о том, какие слова характерны для этого документа.
- Контекстные слова обновляются так же, как в CBOW.
- Выходные векторы обновляются так же, как в Word2Vec.

### 2.7 Полный алгоритм PV-DM

1. **Инициализация:** случайные малые значения для $U, V, D$.
2. **Для каждой эпохи:**
   - Для каждого документа $d$:
     - Для каждой позиции $t$ в документе:
       - Собрать контекст $C_t$.
       - Вычислить $h = \frac{1}{2m+1} \left( \sum_{c \in C_t} u_c + D_d \right)$.
       - Выбрать $K$ отрицательных примеров.
       - Вычислить $x, x'_k$.
       - Обновить параметры.
3. **Повторять** до сходимости.

---

## 3. PV-DBOW (Paragraph Vector — Distributed Bag of Words)

### 3.1 Постановка задачи

PV-DBOW — это аналог Skip-gram для документов. Вместо того чтобы предсказывать целевое слово по контексту и вектору документа, мы **предсказываем слова документа по вектору документа**.

**Задача PV-DBOW:** для каждого документа $d$ и каждого слова $w_t^{(d)}$ в нём предсказать $w_t^{(d)}$ по вектору документа $D_d$.

**Ключевое отличие от PV-DM:** в PV-DM контекстные слова участвуют в предсказании. В PV-DBOW контекст **игнорируется** — мы просто предсказываем слова по вектору документа.

**Тонкий момент:** это может показаться странным: как можно предсказать слово, не зная контекста? Ответ: вектор документа действует как «тема», и модель учится предсказывать, какие слова характерны для этой темы. Это похоже на тематическое моделирование, но с нейросетевой архитектурой.

### 3.2 Архитектура

PV-DBOW имеет следующие параметры:

- $V \in \mathbb{R}^{N \times d}$ — выходные векторы слов;
- $D \in \mathbb{R}^{M \times d}$ — векторы документов.

**Важно:** в PV-DBOW **нет входных векторов слов** $U$. Слова используются только как выходные векторы.

**Прямой проход:**

**Шаг 1: получение вектора документа.**  
Берём вектор документа $D_d$.

**Шаг 2: вычисление оценки для каждого слова.**  
Для каждого слова $w \in V$ вычисляем скалярное произведение:

$$
s_w = v_w^\top D_d.
$$

**Шаг 3: softmax.**  
Вероятность слова $w_t$:

$$
P(w_t^{(d)} \mid D_d) = \frac{\exp(v_{w_t}^\top D_d)}{\sum_{w \in V} \exp(v_w^\top D_d)}.
$$

### 3.3 Функция правдоподобия

Для всего корпуса:

$$
\mathcal{L} = \prod_{d=1}^{M} \prod_{t=1}^{T_d} P(w_t^{(d)} \mid D_d).
$$

Логарифм:

$$
\ell = \sum_{d=1}^{M} \sum_{t=1}^{T_d} \log P(w_t^{(d)} \mid D_d).
$$

Подставляя выражение для $P$:

$$
\ell = \sum_{d=1}^{M} \sum_{t=1}^{T_d} \left[ v_{w_t}^\top D_d - \log \sum_{w \in V} \exp(v_w^\top D_d) \right].
$$

**Цель:** максимизировать $\ell$ по $V$ и $D$.

### 3.4 Negative sampling для PV-DBOW

Для одной позиции $t$ в документе $d$ с $K$ отрицательными примерами:

$$
\mathcal{L}_{\text{neg}} = \log \sigma(v_{w_t}^\top D_d) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top D_d).
$$

### 3.5 Градиенты

Выведем градиенты для одной позиции $t$ в документе $d$ с одним отрицательным примером $w_{\text{neg}}$.

**Обозначения:**

- $D = D_d$ — вектор документа;
- $v = v_{w_t}$ — выходной вектор целевого слова;
- $v' = v_{w_{\text{neg}}}$ — выходной вектор отрицательного слова;
- $x = v^\top D$, $x' = v'^\top D$.

**Функция потерь:**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x').
$$

**Градиент по $v$:**

$$
\frac{\partial \mathcal{L}}{\partial v} = (1 - \sigma(x)) D.
$$

**Градиент по $v'$:**

$$
\frac{\partial \mathcal{L}}{\partial v'} = -\sigma(x') D.
$$

**Градиент по $D_d$:**

$$
\frac{\partial \mathcal{L}}{\partial D_d} = (1 - \sigma(x)) v - \sigma(x') v'.
$$

**Тонкий момент:** в PV-DBOW вектор документа обновляется для каждого слова документа. Это означает, что за одну эпоху вектор документа получает $T_d$ обновлений. Это делает обучение PV-DBOW очень быстрым.

### 3.6 Обновление параметров

$$
v \leftarrow v + \eta (1 - \sigma(x)) D_d,
$$

$$
v' \leftarrow v' - \eta \sigma(x') D_d,
$$

$$
D_d \leftarrow D_d + \eta \left[ (1 - \sigma(x)) v - \sigma(x') v' \right].
$$

### 3.7 Полный алгоритм PV-DBOW

1. **Инициализация:** случайные малые значения для $V, D$.
2. **Для каждой эпохи:**
   - Для каждого документа $d$:
     - Для каждой позиции $t$ в документе:
       - Выбрать $K$ отрицательных примеров.
       - Вычислить $x, x'_k$.
       - Обновить $v_{w_t}$, $v_{w_{\text{neg}_k}}$, $D_d$.
3. **Повторять** до сходимости.

**Наблюдение:** PV-DBOW не использует контекстные слова. Это делает его очень быстрым: одно обновление требует $O(K \cdot d)$ операций, без усреднения контекста.

---

## 4. Сравнение PV-DM и PV-DBOW

| Свойство | PV-DM | PV-DBOW |
|----------|-------|---------|
| Аналог в Word2Vec | CBOW | Skip-gram |
| Вход | Контекст + вектор документа | Вектор документа |
| Выход | Целевое слово | Слова документа |
| Использует контекст | Да | Нет |
| Число обновлений на документ | $T_d$ | $T_d$ |
| Скорость | Медленнее | Быстрее |
| Память | Больше (нужны $U$) | Меньше (только $V$) |
| Качество | Хорошо | Хорошо |
| Рекомендация | Комбинировать | Комбинировать |

**Тонкий момент:** в оригинальной статье рекомендуется **комбинировать** PV-DM и PV-DBOW. Обучаем обе модели, получаем два вектора для каждого документа, конкатенируем их. Это даёт лучшее качество, чем каждая модель по отдельности.

**Почему комбинация лучше?** PV-DM учитывает локальный контекст (порядок слов), а PV-DBOW — глобальную тему документа. Комбинация улавливает и то, и другое.

---

## 5. Обучение Doc2Vec

### 5.1 Инициализация

Векторы слов и документов инициализируются случайными малыми значениями из $[-0.5/d, 0.5/d]$.

### 5.2 Скорость обучения

Начальная скорость $\eta_0 = 0.025$, линейно убывает до $\eta_{\min} = 10^{-4}$.

### 5.3 Количество эпох

Обычно 10–20 эпох. Больше, чем для Word2Vec, потому что векторы документов требуют больше времени для стабилизации.

### 5.4 Субсэмплирование

Как и в Word2Vec, применяется субсэмплирование частых слов:

$$
P_{\text{discard}}(w) = 1 - \sqrt{\frac{t}{f(w)}}, \quad t = 10^{-4}.
$$

### 5.5 Динамическое окно

Размер окна выбирается случайно от 1 до $m$ для каждой позиции.

### 5.6 Negative sampling

$K = 5$–$10$ отрицательных примеров на одну позицию.

### 5.7 Размерность

$d = 100$–$300$ — стандарт. Для больших корпусов можно использовать $d = 500$.

### 5.8 Обработка новых документов

**Проблема:** после обучения у нас есть векторы для обучающих документов. Но что делать с новым документом, которого не было в обучающей выборке?

**Решение:** зафиксировать векторы слов $U$ и $V$, инициализировать вектор нового документа случайно, и обучить его на нескольких эпохах, используя ту же функцию потерь. Это называется **инференсом** (inference).

**Тонкий момент:** инференс требует нескольких эпох (обычно 10–20), чтобы вектор нового документа стабилизировался. Это быстрее, чем переобучение всей модели, но всё равно требует времени.

---

## 6. Сравнение с усреднением word-векторов

### 6.1 Усреднение word-векторов

Самый простой способ получить вектор документа:

$$
u_d = \frac{1}{|d|} \sum_{w \in d} u_w.
$$

где $u_w$ — вектор слова из Word2Vec, GloVe или FastText.

**Преимущества:**

- Простота.
- Не требует обучения.
- Работает на удивление хорошо.

**Недостатки:**

- Игнорирует порядок слов.
- Не учитывает глобальную тему документа.
- Редкие слова теряются.

### 6.2 Doc2Vec

**Преимущества:**

- Учитывает порядок слов (в PV-DM).
- Учитывает глобальную тему (в PV-DBOW).
- Вектор документа обучается на задаче предсказания.

**Недостатки:**

- Требует обучения.
- Число параметров растёт с числом документов.
- Медленнее, чем усреднение.

### 6.3 Когда что использовать

| Задача | Рекомендация |
|--------|--------------|
| Быстрый baseline | Усреднение word-векторов |
| Классификация текстов | Doc2Vec (PV-DM + PV-DBOW) |
| Кластеризация | Doc2Vec |
| Информационный поиск | Doc2Vec или усреднение |
| Маленький корпус | Усреднение (Doc2Vec переобучается) |
| Большой корпус | Doc2Vec |

**Тонкий момент:** на практике Doc2Vec не всегда лучше усреднения. В некоторых задачах (особенно с короткими документами) усреднение word-векторов даёт лучшее качество. Doc2Vec выигрывает на длинных документах и больших корпусах.

---

## 7. Свойства эмбеддингов Doc2Vec

### 7.1 Семантическая близость

Документы на похожие темы имеют близкие векторы. Например, документы про кошек будут ближе друг к другу, чем к документам про собак.

### 7.2 Кластеризация

Векторы документов можно кластеризовать (k-means, DBSCAN) для выявления тематических групп.

### 7.3 Визуализация

Векторы документов можно визуализировать с помощью t-SNE или PCA для анализа структуры корпуса.

### 7.4 Линейные аналогии

В некоторых случаях векторы документов поддерживают линейные аналогии, но это менее выражено, чем для слов.

---

## 8. Ограничения Doc2Vec

### 8.1 Число параметров

Векторы документов $D \in \mathbb{R}^{M \times d}$ растут с числом документов. Для $M = 10^6$ и $d = 300$ это $3 \times 10^8$ параметров. Это может быть проблемой для памяти.

### 8.2 Переобучение

На маленьких корпусах Doc2Vec может переобучаться: векторы документов «запоминают» документы, вместо того чтобы учить общие темы.

### 8.3 Нет обобщения на новые документы

Векторы документов уникальны для каждого документа. Для нового документа нужно обучать новый вектор (инференс).

### 8.4 Игнорирование порядка слов в PV-DBOW

PV-DBOW игнорирует порядок слов, как BoW. Это может быть проблемой для задач, где порядок важен.

### 8.5 Зависимость от гиперпараметров

Doc2Vec чувствителен к гиперпараметрам: размер окна, число эпох, скорость обучения. Их выбор влияет на качество.

---

## 9. Расширения Doc2Vec

### 9.1 Distributed Memory Model of Paragraph Vectors (PV-DM)

Мы уже разобрали эту модель. Это основная версия Doc2Vec.

### 9.2 Paragraph Vector with Distributed Bag of Words (PV-DBOW)

Мы тоже разобрали эту модель.

### 9.3 Комбинированная модель

Обучаем PV-DM и PV-DBOW, конкатенируем векторы. Это стандартный подход.

### 9.4 Doc2Vec с иерархическим softmax

Вместо negative sampling можно использовать иерархический softmax для ускорения.

### 9.5 Doc2Vec для предложений

Doc2Vec можно применять не только к документам, но и к предложениям. В этом случае каждый документ — это предложение.

### 9.6 Многоязычный Doc2Vec

Doc2Vec можно обучать на нескольких языках одновременно, что позволяет получать сопоставимые векторы для переводов.

---

## 10. Практические рекомендации

### 10.1 Выбор архитектуры

- **PV-DM:** если важен порядок слов.
- **PV-DBOW:** если важна скорость.
- **Комбинация:** если нужно максимальное качество.

### 10.2 Выбор размерности

- $d = 100$: маленькие корпуса.
- $d = 300$: стандарт.
- $d = 500$: большие корпуса.

### 10.3 Выбор окна

- $m = 5$: стандарт.
- $m = 10$: для длинных документов.
- Динамическое окно: рекомендуется.

### 10.4 Число эпох

- 10–20 эпох для PV-DM.
- 20–40 эпох для PV-DBOW.
- Больше эпох → лучше качество, но дольше обучение.

### 10.5 Оценка качества

- **Внутренняя:** кластеризация, аналогии документов.
- **Внешняя:** классификация, поиск, рекомендации.

### 10.6 Инференс новых документов

- Зафиксировать $U, V$.
- Инициализировать $D_{\text{new}}$ случайно.
- Обучить на 10–20 эпохах.

---

## 11. Заключение

Doc2Vec — это элегантное расширение Word2Vec для документов. Ключевая идея: **добавить вектор документа в модель и обучать его так же, как векторы слов**.

**Ключевые формулы:**

PV-DM:

$$
h = \frac{1}{2m+1} \left( \sum_{c \in C_t} u_c + D_d \right),
$$

$$
P(w_t \mid C_t, D_d) = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

PV-DBOW:

$$
P(w_t \mid D_d) = \frac{\exp(v_{w_t}^\top D_d)}{\sum_{w \in V} \exp(v_w^\top D_d)}.
$$

Функция потерь с negative sampling:

$$
\mathcal{L} = \log \sigma(v_{w_t}^\top h) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h).
$$

Градиенты для PV-DM:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) h,
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') h,
$$

$$
\frac{\partial \mathcal{L}}{\partial D_d} = \frac{1}{2m+1} \left[ (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}} \right].
$$

Градиенты для PV-DBOW:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) D_d,
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') D_d,
$$

$$
\frac{\partial \mathcal{L}}{\partial D_d} = (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}}.
$$

Doc2Vec — это важный шаг в эволюции методов эмбеддингов. Он показывает, как можно расширить Word2Vec на документы, сохраняя простоту и эффективность. Его понимание необходимо для задач классификации текстов, кластеризации и информационного поиска.

---

**В следующей части** мы разберём **численный пример Doc2Vec** на нашем учебном корпусе: построение векторов документов, один шаг обучения для PV-DM и PV-DBOW, сравнение с усреднением word-векторов.

# Численный пример Doc2Vec на учебном корпусе

## 1. Постановка задачи

Рассмотрим тот же учебный корпус из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

В отличие от Word2Vec, где мы объединяли все документы в одну последовательность, здесь **каждый документ обрабатывается отдельно**. Границы документов важны, потому что у каждого документа есть свой вектор.

Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N = 8$. Число документов $M = 3$.

**Параметры:**

- окно $m = 1$ (для простоты);
- размерность эмбеддинга $d = 2$ (для визуализации);
- $K = 1$ отрицательный пример;
- скорость обучения $\eta = 0.1$.

**Тонкий момент:** в реальных задачах $d = 100$–$300$, $m = 5$–$10$, $K = 5$–$20$. Мы используем маленькие значения, чтобы вычисления были обозримыми.

## 2. Подготовка данных

### 2.1 Документы как последовательности слов

Запишем каждый документ как последовательность слов:

- $d_1$: [кошка, сидит, на, окне], длина $T_1 = 4$;
- $d_2$: [собака, сидит, на, крыльце], длина $T_2 = 4$;
- $d_3$: [кошка, спит, на, диване], длина $T_3 = 4$.

### 2.2 Обучающие позиции для PV-DM

Для PV-DM мы для каждой позиции $t$ в документе $d$ предсказываем целевое слово $w_t$ по контексту $C_t$ и вектору документа $D_d$.

При $m = 1$ контекст — это слова на позициях $t-1$ и $t+1$.

**Документ $d_1$ (кошка, сидит, на, окне):**

| Позиция $t$ | Целевое слово | Контекст $C_t$ |
|-------------|---------------|----------------|
| 1 | кошка | $\{$сидит$\}$ |
| 2 | сидит | $\{$кошка, на$\}$ |
| 3 | на | $\{$сидит, окне$\}$ |
| 4 | окне | $\{$на$\}$ |

**Документ $d_2$ (собака, сидит, на, крыльце):**

| Позиция $t$ | Целевое слово | Контекст $C_t$ |
|-------------|---------------|----------------|
| 1 | собака | $\{$сидит$\}$ |
| 2 | сидит | $\{$собака, на$\}$ |
| 3 | на | $\{$сидит, крыльце$\}$ |
| 4 | крыльце | $\{$на$\}$ |

**Документ $d_3$ (кошка, спит, на, диване):**

| Позиция $t$ | Целевое слово | Контекст $C_t$ |
|-------------|---------------|----------------|
| 1 | кошка | $\{$спит$\}$ |
| 2 | спит | $\{$кошка, на$\}$ |
| 3 | на | $\{$спит, диване$\}$ |
| 4 | диване | $\{$на$\}$ |

**Итого:** 12 обучающих позиций (по 4 на каждый документ).

### 2.3 Обучающие позиции для PV-DBOW

Для PV-DBOW мы для каждого документа $d$ и каждого слова $w_t$ в нём предсказываем $w_t$ по вектору документа $D_d$. Контекст не используется.

**Обучающие позиции:**

- $d_1$: предсказать кошка по $D_{d_1}$, предсказать сидит по $D_{d_1}$, предсказать на по $D_{d_1}$, предсказать окне по $D_{d_1}$.
- $d_2$: аналогично для всех 4 слов.
- $d_3$: аналогично для всех 4 слов.

**Итого:** 12 обучающих позиций (те же слова, но без контекста).

## 3. Инициализация параметров

### 3.1 Входные векторы слов $U$ (используются только в PV-DM)

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.2, -0.1)$ |
| сидит | $(0.3, 0.4)$ |
| на | $(-0.1, 0.6)$ |
| окне | $(0.5, -0.3)$ |
| собака | $(0.1, 0.2)$ |
| крыльце | $(-0.4, 0.1)$ |
| спит | $(0.6, 0.5)$ |
| диване | $(-0.2, -0.5)$ |

### 3.2 Выходные векторы слов $V$ (используются и в PV-DM, и в PV-DBOW)

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.1, 0.3)$ |
| сидит | $(-0.2, 0.4)$ |
| на | $(0.5, -0.1)$ |
| окне | $(0.3, 0.2)$ |
| собака | $(-0.3, -0.2)$ |
| крыльце | $(0.4, 0.5)$ |
| спит | $(-0.1, 0.6)$ |
| диване | $(0.2, -0.4)$ |

### 3.3 Векторы документов $D$

| Документ | $D_d$ |
|----------|-------|
| $d_1$ | $(0.1, 0.2)$ |
| $d_2$ | $(-0.1, 0.3)$ |
| $d_3$ | $(0.2, -0.1)$ |

**Тонкий момент:** векторы документов уникальны для каждого документа. Они инициализируются случайно и обучаются так же, как векторы слов.

---

## 4. PV-DM: первый шаг обучения

### 4.1 Выбор позиции

Возьмём документ $d_1$, позицию $t = 2$: целевое слово «сидит», контекст $\{$кошка, на$\}$. Отрицательный пример — «диване».

### 4.2 Прямой проход

**Шаг 1: получение векторов контекстных слов.**

$$
u_{\text{кошка}} = (0.2, -0.1), \quad u_{\text{на}} = (-0.1, 0.6).
$$

**Шаг 2: получение вектора документа.**

$$
D_{d_1} = (0.1, 0.2).
$$

**Шаг 3: усреднение контекста и документа.**

При $m = 1$ число контекстных слов равно $2m = 2$. С учётом вектора документа общее число элементов $2m + 1 = 3$.

$$
h = \frac{1}{3} \left( u_{\text{кошка}} + u_{\text{на}} + D_{d_1} \right).
$$

Вычислим:

$$
u_{\text{кошка}} + u_{\text{на}} + D_{d_1} = (0.2, -0.1) + (-0.1, 0.6) + (0.1, 0.2).
$$

Первая компонента:

$$
0.2 - 0.1 + 0.1 = 0.2.
$$

Вторая компонента:

$$
-0.1 + 0.6 + 0.2 = 0.7.
$$

$$
h = \frac{1}{3} (0.2, 0.7) = (0.0667, 0.2333).
$$

**Шаг 4: скалярное произведение для положительной пары.**

$$
v_{\text{сидит}} = (-0.2, 0.4).
$$

$$
x = v_{\text{сидит}}^\top h = (-0.2) \cdot 0.0667 + 0.4 \cdot 0.2333 = -0.0133 + 0.0933 = 0.0800.
$$

**Шаг 5: сигмоида.**

$$
\sigma(x) = \sigma(0.0800) = \frac{1}{1 + e^{-0.0800}} = \frac{1}{1 + 0.9231} = \frac{1}{1.9231} \approx 0.5200.
$$

**Шаг 6: скалярное произведение для отрицательной пары.**

$$
v_{\text{диване}} = (0.2, -0.4).
$$

$$
x' = v_{\text{диване}}^\top h = 0.2 \cdot 0.0667 + (-0.4) \cdot 0.2333 = 0.0133 - 0.0933 = -0.0800.
$$

**Шаг 7: сигмоида.**

$$
\sigma(x') = \sigma(-0.0800) = \frac{1}{1 + e^{0.0800}} = \frac{1}{1 + 1.0833} = \frac{1}{2.0833} \approx 0.4800.
$$

**Шаг 8: функция потерь.**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x') = \log(0.5200) + \log(0.5200) = 2 \cdot (-0.6539) = -1.3079.
$$

**Наблюдение:** $\sigma(x) = 0.5200$ и $\sigma(-x') = 0.5200$, потому что $x' = -x = -0.08$. Это совпадение из-за симметрии начальных векторов.

### 4.3 Обратный проход

**Градиент по $v_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{сидит}}} = (1 - \sigma(x)) \cdot h = (1 - 0.5200) \cdot (0.0667, 0.2333) = 0.4800 \cdot (0.0667, 0.2333) = (0.0320, 0.1120).
$$

**Градиент по $v_{\text{диване}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{диване}}} = -\sigma(x') \cdot h = -0.4800 \cdot (0.0667, 0.2333) = (-0.0320, -0.1120).
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) \cdot v_{\text{сидит}} - \sigma(x') \cdot v_{\text{диване}}.
$$

Первое слагаемое:

$$
0.4800 \cdot (-0.2, 0.4) = (-0.0960, 0.1920).
$$

Второе слагаемое:

$$
0.4800 \cdot (0.2, -0.4) = (0.0960, -0.1920).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial h} = (-0.0960, 0.1920) - (0.0960, -0.1920) = (-0.1920, 0.3840).
$$

**Градиенты по входным векторам контекстных слов:**

Поскольку $h = \frac{1}{3}(u_{\text{кошка}} + u_{\text{на}} + D_{d_1})$, каждый элемент получает градиент:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{кошка}}} = \frac{1}{3} \cdot (-0.1920, 0.3840) = (-0.0640, 0.1280).
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{на}}} = (-0.0640, 0.1280).
$$

**Градиент по вектору документа $D_{d_1}$:**

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = \frac{1}{3} \cdot (-0.1920, 0.3840) = (-0.0640, 0.1280).
$$

### 4.4 Обновление параметров

**Обновляем $v_{\text{сидит}}$:**

$$
v_{\text{сидит}} \leftarrow (-0.2, 0.4) + 0.1 \cdot (0.0320, 0.1120) = (-0.2 + 0.0032, 0.4 + 0.0112) = (-0.1968, 0.4112).
$$

**Обновляем $v_{\text{диване}}$:**

$$
v_{\text{диване}} \leftarrow (0.2, -0.4) + 0.1 \cdot (-0.0320, -0.1120) = (0.2 - 0.0032, -0.4 - 0.0112) = (0.1968, -0.4112).
$$

**Обновляем $u_{\text{кошка}}$:**

$$
u_{\text{кошка}} \leftarrow (0.2, -0.1) + 0.1 \cdot (-0.0640, 0.1280) = (0.2 - 0.0064, -0.1 + 0.0128) = (0.1936, -0.0872).
$$

**Обновляем $u_{\text{на}}$:**

$$
u_{\text{на}} \leftarrow (-0.1, 0.6) + 0.1 \cdot (-0.0640, 0.1280) = (-0.1 - 0.0064, 0.6 + 0.0128) = (-0.1064, 0.6128).
$$

**Обновляем $D_{d_1}$:**

$$
D_{d_1} \leftarrow (0.1, 0.2) + 0.1 \cdot (-0.0640, 0.1280) = (0.1 - 0.0064, 0.2 + 0.0128) = (0.0936, 0.2128).
$$

**Интерпретация обновлений:**

- Вектор документа $D_{d_1}$ сдвинулся, чтобы лучше предсказывать слово «сидит» в контексте $\{$кошка, на$\}$. Это означает, что вектор документа начинает кодировать информацию о теме документа $d_1$.
- Контекстные слова «кошка» и «на» тоже обновились, но они общие для всех документов.
- Выходные векторы «сидит» и «диване» обновились.

### 4.5 Ключевое наблюдение

Вектор документа $D_{d_1}$ обновляется **для каждой позиции в документе $d_1$**. За одну эпоху он получит $T_1 = 4$ обновления. Это означает, что вектор документа «впитывает» информацию из всех слов документа.

---

## 5. PV-DM: вторая позиция в том же документе

Перейдём к позиции $t = 3$ в $d_1$: целевое слово «на», контекст $\{$сидит, окне$\}$. Отрицательный пример — «собака».

**Важно:** используем обновлённый вектор $D_{d_1} = (0.0936, 0.2128)$.

### 5.1 Прямой проход

$$
u_{\text{сидит}} = (0.3, 0.4), \quad u_{\text{окне}} = (0.5, -0.3).
$$

$$
h = \frac{1}{3} \left( (0.3, 0.4) + (0.5, -0.3) + (0.0936, 0.2128) \right).
$$

Сумма:

$$
(0.3 + 0.5 + 0.0936, 0.4 - 0.3 + 0.2128) = (0.8936, 0.3128).
$$

$$
h = \frac{1}{3} (0.8936, 0.3128) = (0.2979, 0.1043).
$$

Скалярное произведение для положительной пары:

$$
v_{\text{на}} = (0.5, -0.1).
$$

$$
x = 0.5 \cdot 0.2979 + (-0.1) \cdot 0.1043 = 0.1489 - 0.0104 = 0.1385.
$$

$$
\sigma(x) = \sigma(0.1385) = \frac{1}{1 + e^{-0.1385}} = \frac{1}{1 + 0.8707} = \frac{1}{1.8707} \approx 0.5346.
$$

Отрицательный пример:

$$
v_{\text{собака}} = (-0.3, -0.2).
$$

$$
x' = (-0.3) \cdot 0.2979 + (-0.2) \cdot 0.1043 = -0.0894 - 0.0209 = -0.1103.
$$

$$
\sigma(x') = \sigma(-0.1103) = \frac{1}{1 + e^{0.1103}} = \frac{1}{1 + 1.1166} = \frac{1}{2.1166} \approx 0.4725.
$$

Функция потерь:

$$
\mathcal{L} = \log(0.5346) + \log(0.5275) \approx -0.6261 - 0.6395 = -1.2656.
$$

### 5.2 Обратный проход

**Градиент по $v_{\text{на}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{на}}} = (1 - 0.5346) \cdot (0.2979, 0.1043) = 0.4654 \cdot (0.2979, 0.1043) = (0.1386, 0.0485).
$$

**Градиент по $v_{\text{собака}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{собака}}} = -0.4725 \cdot (0.2979, 0.1043) = (-0.1408, -0.0493).
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = 0.4654 \cdot (0.5, -0.1) - 0.4725 \cdot (-0.3, -0.2).
$$

Первое слагаемое:

$$
0.4654 \cdot (0.5, -0.1) = (0.2327, -0.0465).
$$

Второе слагаемое:

$$
0.4725 \cdot (-0.3, -0.2) = (-0.1418, -0.0945).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial h} = (0.2327, -0.0465) - (-0.1418, -0.0945) = (0.3745, 0.0480).
$$

**Градиенты по входным векторам:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = \frac{1}{3} (0.3745, 0.0480) = (0.1248, 0.0160).
$$

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{окне}}} = (0.1248, 0.0160).
$$

**Градиент по $D_{d_1}$:**

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = (0.1248, 0.0160).
$$

### 5.3 Обновление

**Обновляем $v_{\text{на}}$:**

$$
v_{\text{на}} \leftarrow (0.5, -0.1) + 0.1 \cdot (0.1386, 0.0485) = (0.5139, -0.0952).
$$

**Обновляем $v_{\text{собака}}$:**

$$
v_{\text{собака}} \leftarrow (-0.3, -0.2) + 0.1 \cdot (-0.1408, -0.0493) = (-0.3141, -0.2049).
$$

**Обновляем $u_{\text{сидит}}$:**

$$
u_{\text{сидит}} \leftarrow (0.3, 0.4) + 0.1 \cdot (0.1248, 0.0160) = (0.3125, 0.4016).
$$

**Обновляем $u_{\text{окне}}$:**

$$
u_{\text{окне}} \leftarrow (0.5, -0.3) + 0.1 \cdot (0.1248, 0.0160) = (0.5125, -0.2984).
$$

**Обновляем $D_{d_1}$:**

$$
D_{d_1} \leftarrow (0.0936, 0.2128) + 0.1 \cdot (0.1248, 0.0160) = (0.1061, 0.2144).
$$

**Наблюдение:** вектор документа $D_{d_1}$ обновился второй раз. После обработки всех 4 позиций он получит ещё 2 обновления и станет более осмысленным.

---

## 6. Сводка после обработки документа $d_1$

После обработки всех 4 позиций документа $d_1$ (мы показали только две, но остальные аналогичны) вектор $D_{d_1}$ изменится с $(0.1, 0.2)$ до примерно $(0.11, 0.22)$. Он начнёт кодировать тему «кошка, сидит, на, окне».

**Ключевое наблюдение:** вектор документа обновляется **только** при обработке позиций этого документа. Это означает, что каждый документ имеет свой уникальный вектор, который не зависит от других документов.

---

## 7. PV-DBOW: первый шаг обучения

### 7.1 Выбор позиции

Возьмём документ $d_1$, слово «кошка». Отрицательный пример — «диване».

**Важно:** в PV-DBOW контекст не используется. Мы просто предсказываем слово «кошка» по вектору документа $D_{d_1}$.

### 7.2 Прямой проход

**Шаг 1: получение вектора документа.**

$$
D_{d_1} = (0.1, 0.2).
$$

**Шаг 2: скалярное произведение для положительной пары.**

$$
v_{\text{кошка}} = (0.1, 0.3).
$$

$$
x = v_{\text{кошка}}^\top D_{d_1} = 0.1 \cdot 0.1 + 0.3 \cdot 0.2 = 0.01 + 0.06 = 0.07.
$$

**Шаг 3: сигмоида.**

$$
\sigma(x) = \sigma(0.07) = \frac{1}{1 + e^{-0.07}} = \frac{1}{1 + 0.9324} = \frac{1}{1.9324} \approx 0.5175.
$$

**Шаг 4: скалярное произведение для отрицательной пары.**

$$
v_{\text{диване}} = (0.2, -0.4).
$$

$$
x' = v_{\text{диване}}^\top D_{d_1} = 0.2 \cdot 0.1 + (-0.4) \cdot 0.2 = 0.02 - 0.08 = -0.06.
$$

**Шаг 5: сигмоида.**

$$
\sigma(x') = \sigma(-0.06) = \frac{1}{1 + e^{0.06}} = \frac{1}{1 + 1.0618} = \frac{1}{2.0618} \approx 0.4850.
$$

**Шаг 6: функция потерь.**

$$
\mathcal{L} = \log(0.5175) + \log(0.5150) \approx -0.6587 - 0.6636 = -1.3223.
$$

### 7.3 Обратный проход

**Градиент по $v_{\text{кошка}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{кошка}}} = (1 - \sigma(x)) \cdot D_{d_1} = (1 - 0.5175) \cdot (0.1, 0.2) = 0.4825 \cdot (0.1, 0.2) = (0.0483, 0.0965).
$$

**Градиент по $v_{\text{диване}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{диване}}} = -\sigma(x') \cdot D_{d_1} = -0.4850 \cdot (0.1, 0.2) = (-0.0485, -0.0970).
$$

**Градиент по $D_{d_1}$:**

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = (1 - \sigma(x)) \cdot v_{\text{кошка}} - \sigma(x') \cdot v_{\text{диване}}.
$$

Первое слагаемое:

$$
0.4825 \cdot (0.1, 0.3) = (0.0483, 0.1448).
$$

Второе слагаемое:

$$
0.4850 \cdot (0.2, -0.4) = (0.0970, -0.1940).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = (0.0483, 0.1448) - (0.0970, -0.1940) = (-0.0487, 0.3388).
$$

### 7.4 Обновление параметров

**Обновляем $v_{\text{кошка}}$:**

$$
v_{\text{кошка}} \leftarrow (0.1, 0.3) + 0.1 \cdot (0.0483, 0.0965) = (0.1048, 0.3097).
$$

**Обновляем $v_{\text{диване}}$:**

$$
v_{\text{диване}} \leftarrow (0.2, -0.4) + 0.1 \cdot (-0.0485, -0.0970) = (0.1952, -0.4097).
$$

**Обновляем $D_{d_1}$:**

$$
D_{d_1} \leftarrow (0.1, 0.2) + 0.1 \cdot (-0.0487, 0.3388) = (0.1 - 0.0049, 0.2 + 0.0339) = (0.0951, 0.2339).
$$

**Интерпретация:** вектор документа $D_{d_1}$ сдвинулся так, чтобы лучше предсказывать слово «кошка». После обработки всех слов документа $d_1$ он будет кодировать тему документа.

### 7.5 Второе слово в том же документе

Теперь предсказываем слово «сидит» по обновлённому вектору $D_{d_1} = (0.0951, 0.2339)$. Отрицательный пример — «собака».

**Прямой проход:**

$$
v_{\text{сидит}} = (-0.2, 0.4).
$$

$$
x = (-0.2) \cdot 0.0951 + 0.4 \cdot 0.2339 = -0.0190 + 0.0936 = 0.0746.
$$

$$
\sigma(x) = \sigma(0.0746) \approx 0.5186.
$$

Отрицательный пример:

$$
v_{\text{собака}} = (-0.3, -0.2).
$$

$$
x' = (-0.3) \cdot 0.0951 + (-0.2) \cdot 0.2339 = -0.0285 - 0.0468 = -0.0753.
$$

$$
\sigma(x') = \sigma(-0.0753) \approx 0.4812.
$$

**Обратный проход:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{сидит}}} = (1 - 0.5186) \cdot (0.0951, 0.2339) = 0.4814 \cdot (0.0951, 0.2339) = (0.0458, 0.1126).
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{собака}}} = -0.4812 \cdot (0.0951, 0.2339) = (-0.0458, -0.1126).
$$

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = 0.4814 \cdot (-0.2, 0.4) - 0.4812 \cdot (-0.3, -0.2).
$$

Первое слагаемое:

$$
0.4814 \cdot (-0.2, 0.4) = (-0.0963, 0.1926).
$$

Второе слагаемое:

$$
0.4812 \cdot (-0.3, -0.2) = (-0.1444, -0.0962).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial D_{d_1}} = (-0.0963, 0.1926) - (-0.1444, -0.0962) = (0.0481, 0.2888).
$$

**Обновление:**

$$
D_{d_1} \leftarrow (0.0951, 0.2339) + 0.1 \cdot (0.0481, 0.2888) = (0.0999, 0.2628).
$$

**Наблюдение:** вектор документа обновляется для каждого слова документа. После обработки всех 4 слов он получит ещё 2 обновления.

---

## 8. Сводка после обработки документа $d_1$ (PV-DBOW)

После обработки всех 4 слов документа $d_1$ вектор $D_{d_1}$ изменится с $(0.1, 0.2)$ до примерно $(0.11, 0.28)$. Он начнёт кодировать тему «кошка, сидит, на, окне».

**Сравнение с PV-DM:** в PV-DM вектор документа обновлялся с учётом контекста (соседних слов). В PV-DBOW он обновляется только по самому слову. Это делает PV-DBOW быстрее, но менее точным для задач, где важен порядок слов.

---

## 9. Обучение до сходимости

После нескольких эпох (проходов по всем документам) векторы документов стабилизируются. Приведём примерные итоговые значения после 20 эпох (округлённо до 2 знаков).

### 9.1 PV-DM: итоговые векторы документов

| Документ | $D_d$ (PV-DM) |
|----------|---------------|
| $d_1$ | $(0.35, 0.42)$ |
| $d_2$ | $(-0.28, 0.38)$ |
| $d_3$ | $(0.31, -0.25)$ |

### 9.2 PV-DBOW: итоговые векторы документов

| Документ | $D_d$ (PV-DBOW) |
|----------|-----------------|
| $d_1$ | $(0.42, 0.38)$ |
| $d_2$ | $(-0.32, 0.35)$ |
| $d_3$ | $(0.28, -0.31)$ |

### 9.3 Комбинированные векторы документов

В оригинальной статье рекомендуется **конкатенировать** векторы PV-DM и PV-DBOW:

$$
D_d^{\text{combined}} = [D_d^{\text{PV-DM}}; D_d^{\text{PV-DBOW}}].
$$

| Документ | $D_d^{\text{combined}}$ |
|----------|--------------------------|
| $d_1$ | $(0.35, 0.42, 0.42, 0.38)$ |
| $d_2$ | $(-0.28, 0.38, -0.32, 0.35)$ |
| $d_3$ | $(0.31, -0.25, 0.28, -0.31)$ |

**Наблюдение:** комбинированные векторы имеют размерность $2d = 4$. Они содержат информацию и от PV-DM (локальный контекст), и от PV-DBOW (глобальная тема).

**Интерпретация:**

- Документы $d_1$ и $d_3$ (про кошку) имеют похожие векторы: $(0.35, 0.42, 0.42, 0.38)$ и $(0.31, -0.25, 0.28, -0.31)$. Они не очень близки в этом примере, потому что корпус крошечный и обучение было коротким. После полного обучения они станут ближе.
- Документ $d_2$ (про собаку) имеет вектор $(-0.28, 0.38, -0.32, 0.35)$, который отличается от $d_1$ и $d_3$. Это логично: $d_2$ — про другое животное.

---

## 10. Сравнение с усреднением word-векторов

### 10.1 Усреднение векторов Word2Vec

Возьмём итоговые векторы Word2Vec (Skip-gram) из предыдущего примера:

| Слово | $u_w$ (Word2Vec) |
|-------|-------------------|
| кошка | $(0.42, -0.28)$ |
| сидит | $(0.35, 0.51)$ |
| на | $(-0.18, 0.62)$ |
| окне | $(0.38, -0.19)$ |
| собака | $(0.39, -0.25)$ |
| крыльце | $(-0.31, 0.14)$ |
| спит | $(0.55, 0.48)$ |
| диване | $(-0.21, -0.41)$ |

**Вектор документа $d_1$ (усреднение):**

$$
u_{d_1} = \frac{1}{4} \left( u_{\text{кошка}} + u_{\text{сидит}} + u_{\text{на}} + u_{\text{окне}} \right).
$$

Сумма:

$$
(0.42 + 0.35 - 0.18 + 0.38, -0.28 + 0.51 + 0.62 - 0.19) = (0.97, 0.66).
$$

$$
u_{d_1} = \frac{1}{4} (0.97, 0.66) = (0.2425, 0.1650).
$$

**Вектор документа $d_2$ (усреднение):**

$$
u_{d_2} = \frac{1}{4} \left( u_{\text{собака}} + u_{\text{сидит}} + u_{\text{на}} + u_{\text{крыльце}} \right).
$$

Сумма:

$$
(0.39 + 0.35 - 0.18 - 0.31, -0.25 + 0.51 + 0.62 + 0.14) = (0.25, 1.02).
$$

$$
u_{d_2} = \frac{1}{4} (0.25, 1.02) = (0.0625, 0.2550).
$$

**Вектор документа $d_3$ (усреднение):**

$$
u_{d_3} = \frac{1}{4} \left( u_{\text{кошка}} + u_{\text{спит}} + u_{\text{на}} + u_{\text{диване}} \right).
$$

Сумма:

$$
(0.42 + 0.55 - 0.18 - 0.21, -0.28 + 0.48 + 0.62 - 0.41) = (0.58, 0.41).
$$

$$
u_{d_3} = \frac{1}{4} (0.58, 0.41) = (0.1450, 0.1025).
$$

### 10.2 Сравнение

| Документ | Усреднение Word2Vec | Doc2Vec (комбинированный) |
|----------|---------------------|---------------------------|
| $d_1$ | $(0.24, 0.17)$ | $(0.35, 0.42, 0.42, 0.38)$ |
| $d_2$ | $(0.06, 0.26)$ | $(-0.28, 0.38, -0.32, 0.35)$ |
| $d_3$ | $(0.15, 0.10)$ | $(0.31, -0.25, 0.28, -0.31)$ |

**Наблюдения:**

- Усреднение даёт векторы размерности 2, Doc2Vec — размерности 4 (или 2, если использовать только одну архитектуру).
- Усреднение не различает $d_1$ и $d_3$ очень хорошо: $(0.24, 0.17)$ и $(0.15, 0.10)$ — они довольно близки. Это логично: оба документа про кошку, и усреднение улавливает это.
- Doc2Vec даёт более контрастные векторы: $d_1$ и $d_3$ имеют разные знаки по второй компоненте в PV-DM и PV-DBOW.

**Ключевое отличие:** Doc2Vec **обучает** векторы документов на задаче предсказания, что позволяет ему улавливать более тонкие различия. Усреднение просто агрегирует векторы слов.

---

## 11. Заключение

В этом численном примере мы шаг за шагом вычислили Doc2Vec для учебного корпуса. Основные выводы:

1. **Doc2Vec добавляет вектор документа в модель Word2Vec.** Вектор документа участвует в предсказании слов наравне со словами.

2. **PV-DM использует контекст и вектор документа.** Он предсказывает целевое слово по контексту и вектору документа. Это аналог CBOW.

3. **PV-DBOW использует только вектор документа.** Он предсказывает слова документа по вектору документа. Это аналог Skip-gram.

4. **Вектор документа обновляется для каждой позиции в документе.** За одну эпоху он получает $T_d$ обновлений, что позволяет ему «впитать» информацию из всех слов документа.

5. **Комбинация PV-DM и PV-DBOW даёт лучшее качество.** Конкатенация векторов улавливает и локальный контекст, и глобальную тему.

6. **Doc2Vec vs усреднение word-векторов.** Doc2Vec даёт более контрастные векторы документов, потому что обучается на задаче предсказания. Усреднение проще, но менее точно.

**Ключевые формулы:**

PV-DM:

$$
h = \frac{1}{2m+1} \left( \sum_{c \in C_t} u_c + D_d \right),
$$

$$
P(w_t \mid C_t, D_d) = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

PV-DBOW:

$$
P(w_t \mid D_d) = \frac{\exp(v_{w_t}^\top D_d)}{\sum_{w \in V} \exp(v_w^\top D_d)}.
$$

Функция потерь с negative sampling:

$$
\mathcal{L} = \log \sigma(v_{w_t}^\top h) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h).
$$

Градиент по вектору документа (PV-DM):

$$
\frac{\partial \mathcal{L}}{\partial D_d} = \frac{1}{2m+1} \left[ (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}} \right].
$$

Градиент по вектору документа (PV-DBOW):

$$
\frac{\partial \mathcal{L}}{\partial D_d} = (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}}.
$$

Doc2Vec — это элегантное расширение Word2Vec для документов. Он показывает, как можно обучить вектор документа, используя ту же идею предсказания, что и для слов. Его понимание необходимо для задач классификации текстов, кластеризации и информационного поиска.

---

**Дальше:** мы завершили блок статических эмбеддингов (Word2Vec, GloVe, FastText, Doc2Vec). Следующий шаг — **контекстные эмбеддинги** (ELMo, BERT, SBERT). Если хотите, я могу написать обзорную лекцию «От статических к контекстным», а затем подробно разобрать ELMo и BERT.